# Introducción a Lightning y el Perfilado de Rendimiento (Performance Profiling)

Cuando trabajas con PyTorch, a menudo pasas una cantidad significativa de tiempo escribiendo código repetitivo (boilerplate) para los bucles de entrenamiento, el manejo de datos y la gestión de dispositivos. Este trabajo repetitivo puede distraerte del objetivo principal: diseñar y entrenar tu modelo.

Aquí es donde entra **[Lightning](https://lightning.ai/) (anteriormente PyTorch Lightning)**. Es un framework de alto nivel que organiza tu código de PyTorch y automatiza la ingeniería, permitiéndote concentrarte en la investigación. Este laboratorio te introducirá a la estructura de Lightning y te mostrará cómo simplifica tareas avanzadas como el ajuste de rendimiento.

También conocerás el **Perfilado (Profiling)**, una técnica utilizada para analizar el rendimiento de tu código y encontrar "cuellos de botella" que ralentizan tu entrenamiento. Al final de este laboratorio, habrás utilizado Lightning para diagnosticar y solucionar un problema de rendimiento real, obteniendo un flujo de trabajo completo para construir modelos más eficientes.

Específicamente, vas a:

* Organizar código estándar de PyTorch en los componentes `LightningDataModule` y `LightningModule` de Lightning.
* Ejecutar un bucle de entrenamiento para establecer una línea base de velocidad y precisión.
* Utilizar el Perfilador (Profiler) integrado para diagnosticar un cuello de botella de "complejidad del modelo".
* Verificar tu solución perfilando un modelo más eficiente y comparando los resultados.
* Evaluar el compromiso (trade-off) final entre la velocidad de entrenamiento y el rendimiento del modelo.

## Imports

In [3]:
import sys
import warnings

# Redirect stderr to a black hole to catch other potential messages
class BlackHole:
    def write(self, message):
        pass
    def flush(self):
        pass
sys.stderr = BlackHole()

# Ignore Python-level UserWarnings
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
import lightning.pytorch as pl
import torch
import torch.nn as nn
import torch.optim as optim
from lightning.pytorch.profilers import PyTorchProfiler
from torch.profiler import schedule
from torch.utils.data import DataLoader
from torchmetrics import Accuracy
from torchvision import datasets, transforms

import helper_utils

torch.set_float32_matmul_precision('medium')
warnings.filterwarnings("ignore", category=UserWarning)

## Definiendo los Datos y el Modelo con Lightning

Con el entorno configurado, es hora de estructurar los componentes centrales de tu proyecto. Lightning ofrece un enfoque organizado al separar el manejo de datos de la lógica del modelo. Definirás estos en los siguientes dos pasos.

### Paso 1: Simplificando la Carga de Datos con el `LightningDataModule`

En tus trabajos anteriores con PyTorch, has visto el pipeline de datos estándar: defines un `Dataset` y luego lo envuelves en un `DataLoader`. Esto a menudo requiere que gestiones instancias separadas de `DataLoader` para tus conjuntos de entrenamiento y validación, lo que puede dispersar tu código de manejo de datos.

Lightning simplifica y organiza todo este proceso encapsulando toda la lógica relacionada con los datos en una única clase reutilizable llamada <code>[LightningDataModule](https://lightning.ai/docs/pytorch/stable/data/datamodule.html)</code>. Esta clase se convierte en el centro neurálgico para obtener, preparar y entregar tus datos, lo que mantiene tu script de entrenamiento principal limpio y enfocado en el modelo.

* Define la clase `CIFAR10DataModule` implementando estos métodos esenciales:
    * **`__init__`**: El constructor donde defines las especificaciones para tu pipeline de datos, como `batch_size`, `num_workers` y las transformaciones (`transforms`) de datos.
    * **`prepare_data()`**: Este método maneja la configuración inicial que se realiza una sola vez, como la descarga del dataset. Lightning garantiza que esto ocurra solo en un único proceso para evitar conflictos.
        * Para esta clase, este método verificará si el dataset **CIFAR10** está presente y lo descargará si no es así.
    * **`setup()`**: Prepara los datos para su uso creando las particiones de los `Dataset` de entrenamiento y validación. El argumento `stage` *podría* usarse para configurar diferentes datos para diferentes etapas (por ejemplo, `'fit'`, `'validate'`, `'test'`). Sin embargo, en este caso, la configuración es la misma para todas las etapas y, por defecto, prepara los datos necesarios para la etapa `'fit'`.
        * Piensa en la etapa **'fit'** como el bucle combinado de entrenamiento y validación. Las etapas **'test'** o **'validate'** son similares a una fase de evaluación independiente en PyTorch estándar.
    * **`train_dataloader()` y `val_dataloader()`**: Estos métodos devuelven las instancias conocidas de `DataLoader` de PyTorch, configuradas con los ajustes de rendimiento esenciales.

In [5]:
class CIFAR10DataModule(pl.LightningDataModule):
    """Un LightningDataModule para el dataset CIFAR10."""

    def __init__(self, data_dir='./data', batch_size=128, num_workers=0):
        """
        Inicializa el DataModule.

        Args:
            data_dir (str): Directorio para almacenar los datos.
            batch_size (int): Número de muestras por lote.
            num_workers (int): Número de subprocesos para la carga de datos.
        """
        # Llama al constructor de la clase padre (LightningDataModule).
        super().__init__()
        # Almacena la ruta del directorio de datos.
        self.data_dir = data_dir
        # Almacena el tamaño del lote para los DataLoaders.
        self.batch_size = batch_size
        # Almacena el número de procesos trabajadores para la carga de datos.
        self.num_workers = num_workers
        # Define una secuencia de transformaciones para aplicar a las imágenes.
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def prepare_data(self):
        """Descarga el dataset CIFAR10 si no está presente."""
        
        # Descarga la partición de entrenamiento de CIFAR10.
        datasets.CIFAR10(self.data_dir, train=True, download=True)
        # Descarga la partición de prueba (test) de CIFAR10.
        datasets.CIFAR10(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        """
        Asigna los datasets de entrenamiento/validación para usar en los dataloaders.

        Args:
            stage (str, opcional): La etapa del entrenamiento (ej. 'fit', 'test').
                                  El Trainer de Lightning requiere este argumento, pero
                                  no se utiliza en esta implementación ya que la lógica
                                  es la misma para todas las etapas. Por defecto es None.
        """
        
        # Crea la instancia del dataset de entrenamiento y aplica las transformaciones.
        self.cifar_train = datasets.CIFAR10(self.data_dir, train=True, transform=self.transform)
        # Crea la instancia del dataset de validación (usando el de prueba) y aplica transformaciones.
        self.cifar_val = datasets.CIFAR10(self.data_dir, train=False, transform=self.transform)
    
    def train_dataloader(self):
        """Devuelve el DataLoader para el conjunto de entrenamiento."""
        # El DataLoader maneja el loteado, el barajado y la carga paralela.
        return DataLoader(self.cifar_train, batch_size=self.batch_size, num_workers=self.num_workers, shuffle=True)

    def val_dataloader(self):
        """Devuelve el DataLoader para el conjunto de validación."""
        # El barajado (shuffling) no es necesario para el conjunto de validación.
        return DataLoader(self.cifar_val, batch_size=self.batch_size, num_workers=self.num_workers)

### Paso 2: Estructurando tu modelo con el `LightningModule`

Ahora que has organizado el manejo de los datos, es momento de definir el modelo en sí. En PyTorch estándar, normalmente defines la arquitectura de tu modelo en una clase que hereda de `nn.Module`. El equivalente en Lightning es el <code>[LightningModule](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html)</code>. Esta clase es donde organizarás todo el código relacionado con tu modelo, desde las definiciones de las capas hasta la lógica de un único paso de entrenamiento.

Al separar la lógica del modelo en el `LightningModule` del motor de entrenamiento, Lightning te permite construir modelos potentes y reutilizables sin preocuparte por la compleja ingeniería que los hace funcionar.

* Define la clase `CIFAR10LightningModule` con estos métodos clave:
    * **`__init__()`**: El constructor donde defines la arquitectura de tu red neuronal, la función de pérdida y cualquier métrica.
    * **`forward()`**: Este método define el paso hacia adelante (forward pass) de tu modelo, exactamente igual que en un módulo de PyTorch estándar.
    * **`training_step()`**: Aquí definirás la lógica para un único lote de entrenamiento. Realizas el paso hacia adelante, calculas la pérdida y registras las métricas. Solo necesitas devolver la pérdida; la automatización de Lightning se encarga de la retropropagación (backpropagation) y la actualización de pesos por ti.
    * **`validation_step()`**: Contiene la misma lógica, pero para tus datos de validación.
    * **`configure_optimizers()`**: En este método, seleccionas y devuelves el optimizador y sus hiperparámetros. Lightning lo utilizará para actualizar los parámetros de tu modelo.

In [6]:
class CIFAR10LightningModule(pl.LightningModule):
    """Un LightningModule flexible para la clasificación de imágenes CIFAR10."""

    def __init__(self,
                 learning_rate=1e-3,
                 weight_decay=0.01,
                 conv_channels=(256, 512, 1024),
                 linear_features=2048,
                 num_classes=10):
        """
        Inicializa el LightningModule con parámetros de capa configurables.

        Args:
            learning_rate: La tasa de aprendizaje para el optimizador.
            weight_decay: El decaimiento de pesos (penalización L2) para el optimizador.
            conv_channels: Una tupla que especifica los canales de salida para cada bloque convolucional.
            linear_features: El número de características en la capa oculta totalmente conectada.
            num_classes: El número de clases de salida para la tarea de clasificación.
        """
        # Llama al constructor de la clase padre.
        super().__init__()
        # Guarda los hiperparámetros pasados al constructor. Esto los hace
        # accesibles mediante `self.hparams` y los registra automáticamente.
        self.save_hyperparameters()
        
        # Calcula el tamaño aplanado de los mapas de características después de la
        # capa final de pooling. Esto es necesario para definir el tamaño de entrada
        # de la primera capa totalmente conectada.
        flattened_size = self.hparams.conv_channels[-1] * 4 * 4
        
        # Define la arquitectura del modelo usando un contenedor secuencial.
        self.model = nn.Sequential(
            nn.Conv2d(3, self.hparams.conv_channels[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(self.hparams.conv_channels[0], self.hparams.conv_channels[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(self.hparams.conv_channels[1], self.hparams.conv_channels[2], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(flattened_size, self.hparams.linear_features),
            nn.ReLU(),
            nn.Linear(self.hparams.linear_features, self.hparams.num_classes)
        )
        
        # Inicializa la función de pérdida.
        self.loss_fn = nn.CrossEntropyLoss()
        
        # Inicializa las métricas para rastrear la precisión (accuracy) en entrenamiento y validación.
        self.train_accuracy = Accuracy(task="multiclass", num_classes=self.hparams.num_classes)
        self.val_accuracy = Accuracy(task="multiclass", num_classes=self.hparams.num_classes)

    def forward(self, x):
        """
        Define el paso hacia adelante (forward pass) del modelo.

        Args:
            x: El tensor de entrada que contiene un lote de imágenes.

        Returns:
            El tensor de salida (logits) del modelo.
        """
        # Pasa la entrada a través del modelo secuencial.
        return self.model(x)

    def training_step(self, batch, batch_idx=None):
        """
        Realiza un único paso de entrenamiento.
    
        Args:
            batch: El lote de datos proveniente del dataloader.
            batch_idx: El índice del lote actual. El Trainer de Lightning requiere este
                       argumento, aunque no se utilice en esta implementación.
        """
        # Desempaqueta el lote en entradas (imágenes) y etiquetas.
        inputs, labels = batch
        # Realiza un paso hacia adelante para obtener las predicciones (logits).
        outputs = self(inputs)
        # Calcula la pérdida.
        loss = self.loss_fn(outputs, labels)

        # Registra la pérdida de entrenamiento al final de cada época.
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        # Actualiza la métrica de precisión de entrenamiento con los resultados del lote actual.
        self.train_accuracy(outputs, labels)
        # Registra la precisión de entrenamiento al final de cada época.
        self.log("train_accuracy", self.train_accuracy, on_step=False, on_epoch=True, prog_bar=True)
        
        # Devuelve la pérdida a Lightning para la retropropagación.
        return loss

    def validation_step(self, batch, batch_idx=None):
        """
        Realiza un único paso de validación.
    
        Args:
            batch: El lote de datos proveniente del dataloader.
            batch_idx: El índice del lote actual.
        """
        # Desempaqueta el lote en entradas (imágenes) y etiquetas.
        inputs, labels = batch
        # Realiza un paso hacia adelante para obtener las predicciones (logits).
        outputs = self(inputs)
        # Calcula la pérdida.
        loss = self.loss_fn(outputs, labels)

        # Registra la pérdida de validación al final de cada época.
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        # Actualiza la métrica de precisión de validación con los resultados del lote actual.
        self.val_accuracy(outputs, labels)
        # Registra la precisión de validación al final de cada época.
        self.log("val_accuracy", self.val_accuracy, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        """
        Configura y devuelve el optimizador del modelo.

        Returns:
            Una instancia del optimizador.
        """
        # Crea y devuelve el optimizador AdamW.
        return optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=self.hparams.weight_decay)

Ahora puedes instanciar las clases que has definido. Este proceso es tan sencillo como crear un modelo o un cargador de datos estándar de PyTorch. La diferencia clave es que estás creando objetos de Lightning que organizan la lógica familiar de PyTorch.

* `dm_loader`: Este es el `CIFAR10DataModule` que alimentará de datos al modelo. Lo configurarás para usar `num_workers=2`.
* `model_baseline`: Este es el `CIFAR10LightningModule` que entrenarás y analizarás.

In [7]:
# Instantiate the DataModule (2 workers).
dm_loader = CIFAR10DataModule(num_workers=2)

# Create an instance of the LightningModule.
model_baseline = CIFAR10LightningModule()

## Una ejecución rápida de entrenamiento

Ahora que has definido e instanciado tu `LightningDataModule` y tu `LightningModule`, puedes verlos en acción.

El código a continuación utiliza una función auxiliar que aprovecha el `Trainer` de Lightning para ejecutar un bucle de entrenamiento completo durante cinco épocas. No te preocupes por los detalles de cómo funciona la función de entrenamiento; eso se cubrirá en material futuro. Por ahora, el objetivo es ver tus componentes de Lightning trabajando juntos para entrenar el modelo.

A medida que ejecutes la celda, **presta atención a cuánto tiempo toma el entrenamiento y a los resultados**.

In [8]:
baseline_results = helper_utils.run_full_training(model_baseline, dm_loader)

print("\nTraining Complete!\n")
print("Final Training Metrics:")

print(f"\tTraining Accuracy:    {baseline_results['train_accuracy']}%")
print(f"\tValidation Accuracy:  {baseline_results['val_accuracy']}%")

--- Running Training For 5 Epochs ---


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Training Complete!

Final Training Metrics:
	Training Accuracy:    86.65%
	Validation Accuracy:  74.17%


## Explorando la complejidad del modelo

¡Buen trabajo! Has ejecutado con éxito un bucle de entrenamiento completo utilizando la estructura organizada de Lightning. Ahora, reflexiona sobre ese proceso. ¿Notaste el tiempo total de entrenamiento? Para un dataset clásico y relativamente pequeño como **CIFAR-10**, pudo haber parecido más lento de lo que esperarías. Esta experiencia te lleva a una pregunta esencial que todo profesional del aprendizaje automático debe hacerse: ¿Por qué tardó tanto y es posible lograr resultados similares de manera más eficiente?

Para encontrar la respuesta, necesitarás investigar las causas potenciales. Un culpable común del entrenamiento lento es el modelo mismo. Esto te lleva naturalmente a cuestionar tu arquitectura específica y preguntar:

> ¿Es este modelo demasiado complejo para el dataset?

Para investigar esto, observa más de cerca la arquitectura de tu modelo base. Por defecto, está configurado con:

* Canales convolucionales: `(256, 512, 1024)`
* Características lineales: `2048`

Se trata de una red profunda y ancha. Sin embargo, el dataset **CIFAR-10** consiste en pequeñas imágenes de 32x32 píxeles. Una arquitectura tan potente podría ser excesiva para esta tarea.

Cuando un modelo es innecesariamente complejo para un dataset determinado, puede crear un **"cuello de botella por complejidad del modelo"**. Esto significa que la GPU dedica una cantidad desproporcionada de tiempo a los cálculos propios del modelo, ralentizando el entrenamiento sin proporcionar un beneficio significativo en la precisión.

Entonces, **¿cómo puedes verificar esta sospecha antes de ejecutar una sesión de entrenamiento completa y costosa en tiempo?** Necesitarás una herramienta para mirar dentro de la ejecución de tu código y ver en qué se está invirtiendo el tiempo.

## Profiling: Comprendiendo el rendimiento de tu código

La sección anterior concluyó que necesitas una herramienta para observar el interior de la ejecución de tu código. Esa herramienta es un **Profiler** (Perfilador).

El perfilado es el proceso de analizar tu código para obtener un desglose detallado de dónde se invierte la mayor parte del tiempo y los recursos, como los ciclos de la CPU, el tiempo de la GPU y la memoria. Este análisis es la clave para confirmar tu hipótesis sobre la complejidad del modelo y encontrar cualquier **cuello de botella** de rendimiento, es decir, las partes específicas de tu código que son desproporcionadamente lentas.

Con el perfilado, puedes responder preguntas importantes sobre la eficiencia de tu código:

* ¿Mi modelo está pasando demasiado tiempo en una operación específica?
* ¿Estoy utilizando mi GPU de manera efectiva o está inactiva esperando datos?
* ¿Hay problemas de memoria que puedan afectar el entrenamiento?
* ¿Dónde debo centrar mis esfuerzos de optimización para obtener el mayor impacto?

Normalmente, perfilas tu código *después* de tener un modelo que funcione, pero *antes* de comprometerte con ejecuciones de entrenamiento largas y costosas. Te ayuda a encontrar y solucionar problemas de rendimiento a tiempo, garantizando que tu entrenamiento sea lo más eficiente posible.

**La ventaja de Lightning**

En PyTorch estándar, tendrías que importar manualmente el perfilador y gestionar su estado dentro del bucle de entrenamiento. Lightning simplifica esto en un único paso limpio. Solo necesitas configurar el perfilador y Lightning lo integra automáticamente en su rutina de ejecución para capturar los datos de rendimiento.

* Tu primer paso es especificar una ubicación donde se guardarán los informes de perfilado generados. Estos informes contendrán todos los datos de rendimiento para tu análisis.

In [9]:
log_dir = "./profiler_output"

### Paso 1: Configurar el `PyTorchProfiler`

A continuación, crearás una instancia del <code>[PyTorchProfiler](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.profilers.PyTorchProfiler.html)</code>. Este objeto te permite controlar exactamente cómo se realiza el perfilado.

* `dirpath` y `filename`: Estos argumentos le indican al perfilador dónde guardar su archivo de salida.

* `schedule`: Este es un parámetro vital que controla la actividad del perfilador para asegurar que solo midas pasos de entrenamiento estables. Define una secuencia:
    * `wait`: Ignora los primeros lotes.
        * Este paso es necesario para omitir las operaciones iniciales de configuración única (como la asignación de memoria) que no forman parte de un paso de entrenamiento normal. A diferencia de las otras fases, el perfilador está completamente **inactivo** durante este tiempo.
    * `warmup`: Ejecuta algunos lotes más para permitir que el hardware se estabilice.
        * Este paso es necesario porque el hardware, como las GPUs, requiere unos cuantos lotes para alcanzar un estado estable de máximo rendimiento. A diferencia de la fase `active`, el perfilador se ejecuta pero **descarta** los datos de rendimiento recopilados aquí.
    * `active`: Comienza a registrar datos de rendimiento para el número especificado de lotes.
        * Esta es la fase principal de medición. Es diferente de las otras dos porque es la única fase donde el perfilador **registra y guarda** los datos de rendimiento para que los analices.
    * `repeat`: Especifica cuántas veces debe ejecutarse este ciclo.

* `profile_memory`: Establecer esto en `True` rastrea la asignación de memoria, lo cual es excelente para identificar operaciones con un alto consumo de memoria.

In [10]:
# Configurar el PyTorch Profiler
profiler = PyTorchProfiler(
    # Establecer el directorio para guardar el informe del perfilador
    dirpath=log_dir,
    # Especificar el nombre del archivo para el informe
    filename="profile_report",
    # Definir el cronograma de perfilado (espera -> calentamiento -> activo)
    # Total de 14 pasos
    schedule=schedule(wait=2, warmup=2, active=10, repeat=1),
    # Habilitar el perfilado del uso de memoria
    profile_memory=True
)

### Paso 2: Inicializar el `Trainer`

Aquí es donde todas las piezas se unen. Ahora inicializarás el <code>[Trainer](https://lightning.ai/docs/pytorch/stable/common/trainer.html)</code>, el motor central de Lightning. Este automatiza todo el bucle de entrenamiento, lo cual aprovecharás aquí para realizar una breve ejecución de diagnóstico. Para cambiar de una ejecución de entrenamiento normal a una de perfilado, simplemente pasas el objeto `profiler` configurado al `Trainer`.

* **`profiler=profiler`**: Este es el paso clave donde vinculas el perfilador al proceso de entrenamiento.
* **`max_steps=14`**: Para esta ejecución de diagnóstico, limitarás el entrenamiento a solo 14 pasos, lo suficiente para cubrir el cronograma del perfilador (`wait=2` + `warmup=2` + `active=10`).
* **`accelerator="auto"`**: Este parámetro le indica a Lightning que detecte y utilice automáticamente el hardware disponible (como la GPU).
* **`logger=False` y `enable_model_summary=False`**: Desactivarás el registrador predeterminado y el resumen del modelo para mantener la consola limpia y enfocada en los resultados del perfilador.
* **`enable_checkpointing=False`**: Esto desactiva el guardado automático de puntos de control (checkpoints) del modelo, lo cual no es necesario para una ejecución de diagnóstico corta.

In [11]:
# Inicializar el Trainer
trainer = pl.Trainer(
    # Vincular el perfilador (profiler) configurado
    profiler=profiler,
    # Limitar el entrenamiento a 14 pasos para coincidir con el cronograma del perfilador
    max_steps=14,
    # Seleccionar automáticamente el acelerador de hardware (ej. GPU, CPU)
    accelerator="auto",
    # Usar un solo dispositivo para el entrenamiento
    devices=1,
    # Desactivar el registrador predeterminado para una salida más limpia
    logger=False,
    # Desactivar el resumen del modelo por la misma razón
    enable_model_summary=False,
    # Desactivar el guardado automático de puntos de control (checkpointing)
    enable_checkpointing=False
)

### Step 3: Run the Diagnostic and Profiling

Con el `profiler` y el `Trainer` configurados, y tus objetos de modelo y datos listos, ahora puedes iniciar la ejecución de diagnóstico.

* Realizarás esto llamando a `.fit()` en tu instancia del `Trainer`. Este único comando gestiona todo el bucle de entrenamiento. Debido a que vinculaste el `profiler` al `Trainer`, este comando ejecuta automáticamente la breve sesión de perfilado de diagnóstico en lugar de una sesión de entrenamiento completa.
    * `model_baseline` y `dm_loader`: Estos son los objetos `LightningModule` y `LightningDataModule` que instanciaste en la sección anterior.

In [12]:
# Iniciar la ejecución de entrenamiento y perfilado.
trainer.fit(model_baseline, dm_loader)

# Print a confirmation message when done.
print("\nProfiling Complete!\n")

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Profiling Complete!



### Paso 4: Analizando los resultados del perfilador: De datos brutos a información accionable

El comando `trainer.fit()` que acabas de ejecutar generó un archivo de traza JSON sin procesar. Este archivo contiene un registro sumamente detallado, evento por evento, de cada operación realizada. 
Los datos brutos son muy minuciosos y no necesariamente se centran en el tipo de operación en la que deseas poner énfasis.

Antes de observar la tabla resumida, es útil comprender las **principales categorías de trabajo** que el perfilador está rastreando. 
Durante cualquier paso de entrenamiento, el tiempo se invierte generalmente en algunos aspectos clave:

* **Carga de datos (Data Loading):** Mover un lote de datos desde tu dataloader al dispositivo activo (por ejemplo, la GPU).

* **Computación del modelo (Model Computation):** El trabajo matemático central de tu red. Esto incluye el **paso hacia adelante** (forward pass, pasar datos a través de las capas) y el **paso hacia atrás** (backward pass, calcular los gradientes).

* **Paso del optimizador (Optimizer Step):** Aplicar los gradientes calculados para actualizar los pesos de tu modelo.

* **Sobrecarga del framework (Framework Overhead):** Las operaciones internas ejecutadas por PyTorch y Lightning para coordinar todo el proceso.

Aunque un análisis profundo está fuera del alcance de este laboratorio, la función `display_profiler_logs` en la siguiente celda analiza todos los eventos granulares y te ayuda a ver cuál de estas categorías está consumiendo más tiempo. Presenta un resumen sencillo y ordenado de las operaciones más costosas.

* Ejecuta la siguiente celda para mostrar las 10 operaciones que más tiempo consumieron en tu ejecución de diagnóstico. La tabla está ordenada por tiempo total de forma descendente.
    * Siéntete libre de cambiar el valor de `head` si deseas ver más o menos filas.

In [13]:
# Mostrar las 10 operaciones más costosas en tiempo del informe del perfilador
helper_utils.display_profiler_logs(profiler, head=10)

Row,Operation Sequence,Action,Total Time (ms),Calls
1,2,ProfilerStep*,442.274179,10
2,131,[pl][profile][Strategy]SingleDeviceStrategy.batch_to_device,341.884728,10
3,10,aten::copy_,341.698075,190
4,60,aten::to,341.204892,370
5,133,aten::_to_copy,340.976356,80
6,132,[pl][profile][LightningModule]CIFAR10LightningModule.transfer_batch_to_device,340.651866,10
7,11,cudaMemcpyAsync,340.468044,140
8,152,[pl][profile]run_training_batch,61.954455,10
9,153,[pl][profile][LightningModule]CIFAR10LightningModule.optimizer_step,61.249327,10
10,154,Optimizer.step#AdamW.step,60.934664,10


#### Resumen de la tabla del perfilador

Una visión general de alto nivel de la tabla del perfilador revela las principales categorías de operaciones que se están rastreando:

- **Operaciones ATen (`aten::...`)** : Funciones de tensores de bajo nivel de PyTorch provenientes del backend de C++ ATen:
  - `aten::copy_`
  - `aten::to`
  - `aten::_to_copy`
>  
- **Utilidad de Lightning** : Funciones de Lightning que gestionan la asignación en el dispositivo:
  - `transfer_batch_to_device`
>
- **Runtime de CUDA** : Llamadas al controlador/librerías de la GPU para la transferencia de datos:
  - `cudaMemcpyAsync`
>
- **Lógica de entrenamiento** : Operaciones de alto nivel en el bucle de entrenamiento:
  - `ProfilerStep*` (el envoltorio para todo el paso)
  - `Optimizer.step#AdamW.step` (la actualización del optimizador)

#### Centrando tu investigación en los cálculos del modelo

Como puedes ver, el informe es muy detallado, mezclando cálculos del modelo de alto nivel con transferencias de datos de bajo nivel y sobrecarga del framework. Para probar la hipótesis sobre la complejidad del modelo, necesitas filtrar este ruido y centrar tu atención en los cálculos del modelo.

* Ejecuta la siguiente celda para mostrar una tabla filtrada que presenta el tiempo total de `ProfilerStep*` (la duración completa de una iteración de entrenamiento, incluyendo el paso hacia adelante, hacia atrás y la optimización) junto con las cuatro operaciones que más tiempo consumen.

In [14]:
# Display a focused summary of the profiler report for the baseline run.
# This filters for the overall time and the top 4 computational operations.
helper_utils.display_model_computation_logs(profiler)

Row,Operation Sequence,Action,Total Time (ms),Calls
1,2,ProfilerStep*,442.274179,10
2,59,aten::convolution_backward,4.037066,30
3,33,autograd::engine::evaluate_function: AddmmBackward0,3.754525,20
4,159,aten::conv2d,3.324034,30
5,34,AddmmBackward0,2.653220,20


<br>

**Interpretación de los resultados base: El cuello de botella**

Al observar la tabla filtrada que acabas de generar, puedes ver exactamente dónde pasa el modelo la mayor parte de su tiempo computacional. 
La entrada `ProfilerStep*` representa el tiempo total de un solo paso de entrenamiento. 
Debajo de ella, puedes ver las operaciones matemáticas más costosas: `aten::conv2d` (convolución) y las operaciones del paso hacia atrás (backward pass) para las capas del modelo.

Estas operaciones son los bloques fundamentales de tu red. Su alto costo en esta ejecución lleva a una pregunta importante: **¿qué posibles mejoras podrías realizar?**

Es probable que una arquitectura de este tamaño sea más potente de lo necesario para este dataset. El perfilador muestra que el pesado trabajo computacional del modelo en sí mismo es el factor dominante que ralentiza las cosas. Cuanto más complejos son estos cálculos, más tiempo permanece ocupada la GPU con cada lote. 

A continuación, perfilarás un segundo modelo más optimizado para medir cómo la simplificación de la arquitectura impacta en el rendimiento.

## Perfilado de un modelo más eficiente

El análisis de la sección anterior sugirió un cuello de botella por **complejidad del modelo**. Es probable que la arquitectura de tu modelo base sea demasiado potente para el sencillo dataset CIFAR-10, lo que provoca que sea innecesariamente lento.

Para probar esta hipótesis, ahora perfilarás una segunda versión más simplificada del modelo para medir cómo impacta la simplificación de la arquitectura en el rendimiento.

### Paso 1: Configurar un nuevo perfilador

**Tu tarea**

* Tu primer paso es configurar un nuevo `PyTorchProfiler` para esta segunda ejecución de diagnóstico. La configuración es idéntica a la ejecución de la línea base, pero debes proporcionar un nuevo `filename`. Este es un paso importante para asegurarte de no sobrescribir los resultados de tu primer análisis.
    * `dirpath`: Establécelo en la variable `log_dir`.
    * `filename`: Dale un nuevo nombre, por ejemplo, `"profile_report_efficient"`.
    * `schedule`: Usa el mismo cronograma que el perfilador base (`wait=2`, `warmup=2`, `active=10`, `repeat=1`).
    * `profile_memory`: Habilita esto estableciéndolo en `True`.

In [16]:
try:
    # Configurar el PyTorch Profiler para la ejecución del modelo eficiente
    profiler_efficient = PyTorchProfiler( ### Añade tu código aquí
        
        # Establecer el directorio para guardar el informe del perfilador
        dirpath=log_dir, ### Añade tu código aquí
        
        # Especificar un nuevo nombre de archivo para el informe
        filename="profile_report_efficient", ### Añade tu código aquí
        
        # Definir el cronograma de perfilado (espera -> calentamiento -> activo)
        schedule= schedule(wait = 2, warmup = 2, active = 10, repeat = 1), ### Añade tu código aquí
        
        # Habilitar el perfilado del uso de memoria
        profile_memory=True ### Añade tu código aquí
    )

    print("\033[92mPyTorchProfiler configurado con éxito!")

except Exception as e:
    print("\033[91mAlgo salió mal, ¡inténtalo de nuevo!")
    raise e

PyTorchProfiler configurado con éxito!


<br>
<details>
<summary><span style="color:green;"><strong>Solution (Click here to expand)</strong></span></summary>

```python
# Configurar el PyTorch Profiler para la ejecución del modelo eficiente
profiler_efficient = PyTorchProfiler( ### Añade tu código aquí
    
    # Establecer el directorio para guardar el informe del perfilador
    dirpath=log_dir, ### Añade tu código aquí
    
    # Especificar un nuevo nombre de archivo para el informe
    filename="profile_report_efficient", ### Añade tu código aquí
    
    # Definir el cronograma de perfilado (espera -> calentamiento -> activo)
    schedule=schedule(wait=2, warmup=2, active=10, repeat=1), ### Añade tu código aquí
    
    # Habilitar el perfilado del uso de memoria
    profile_memory=True ### Añade tu código aquí
)
```

### Paso 2: Configurar un nuevo Trainer

**Tu tarea**

* A continuación, crearás una nueva instancia del `Trainer` para esta ejecución. Para garantizar una comparación justa con tu línea base, utilizarás exactamente la misma configuración que antes. El único cambio es pasar tu nuevo objeto `profiler_efficient`.
    * `profiler`: Adjunta el objeto `profiler_efficient` que acabas de crear.
    * `max_steps`: Limita el entrenamiento a `14` pasos para que coincida con el cronograma del perfilador.
    * `accelerator`: Establécelo en `"auto"` para seleccionar automáticamente el hardware.
    * `devices`: Utiliza un solo dispositivo (1).
    * `logger`: Desactiva el registrador estableciéndolo en `False`.
    * `enable_model_summary`: Desactiva el resumen del modelo (`False`).
    * `enable_checkpointing`: Desactiva el guardado automático de puntos de control (`False`).

In [17]:
try:
    # Inicializar el Trainer
    trainer_efficient = pl.Trainer( ### Añade tu código aquí
        
        # Vincular el perfilador configurado
        profiler=profiler_efficient, ### Añade tu código aquí
        
        # Limitar el entrenamiento a 14 pasos para coincidir con el cronograma del perfilador
        max_steps=14, ### Añade tu código aquí
        
        # Seleccionar automáticamente el acelerador de hardware (ej. GPU, CPU)
        accelerator="auto", ### Añade tu código aquí
        
        # Usar un solo dispositivo para el entrenamiento
        devices=1, ### Añade tu código aquí
        
        # Desactivar el registrador predeterminado para una salida más limpia
        logger=False, ### Añade tu código aquí
        
        # Desactivar el resumen del modelo por la misma razón
        enable_model_summary=False, ### Añade tu código aquí

        # Desactivar el guardado automático de puntos de control (checkpointing)
        enable_checkpointing=False ### Añade tu código aquí
    )

    print("\033[92m¡Trainer configurado con éxito!")

except Exception as e:
    print("\033[91m¡Algo salió mal, inténtalo de nuevo!")
    raise e

¡Trainer configurado con éxito!


<br>
<details>
<summary><span style="color:green;"><strong>Solution (Click here to expand)</strong></span></summary>

```python
# Inicializar el Trainer
trainer_efficient = pl.Trainer( ### Añade tu código aquí
    
    # Vincular el perfilador configurado
    profiler=profiler_efficient, ### Añade tu código aquí
    
    # Limitar el entrenamiento a 14 pasos para coincidir con el cronograma del perfilador
    max_steps=14, ### Añade tu código aquí
    
    # Seleccionar automáticamente el acelerador de hardware (ej. GPU, CPU)
    accelerator="auto", ### Añade tu código aquí
    
    # Usar un solo dispositivo para el entrenamiento
    devices=1, ### Añade tu código aquí
    
    # Desactivar el registrador predeterminado para una salida más limpia
    logger=False, ### Añade tu código aquí
    
    # Desactivar el resumen del modelo por la misma razón
    enable_model_summary=False, ### Añade tu código aquí

    # Desactivar el guardado automático de puntos de control (checkpointing)
    enable_checkpointing=False ### Añade tu código aquí
)
)
```

### Paso 3: Perfilar el modelo eficiente

Ahora, instanciarás tu `LightningModule` nuevamente, pero esta vez con una arquitectura mucho más simple (`conv_channels=(32, 64, 128)` y `linear_features=512`), para crear la versión "eficiente" del modelo.

In [18]:
# Create a new instance of the model with a much simpler architecture.
model_efficient = CIFAR10LightningModule(
    conv_channels=(32, 64, 128),
    linear_features=512
)

**Tu tarea**

Ahora es el momento de iniciar la segunda ejecución de diagnóstico. Lo harás llamando al método `.fit()` en tu nueva instancia de `trainer_efficient`.

Debes utilizar el mismo módulo de datos (`dm_loader`) que antes. Esta es una parte importante del experimento, ya que garantiza que la única variable que has cambiado es la arquitectura del modelo.

* Llama a `.fit()` en el objeto `trainer_efficient`.
* Pasa `model_efficient` y `dm_loader` como argumentos.

In [19]:
try:
    # Iniciar la segunda ejecución de diagnóstico con el nuevo modelo optimizado.
    trainer_efficient.fit(model_efficient, dm_loader)### Añade tu código aquí
    
    print("\n¡Perfilado completado!\n")
    
except Exception as e:
    print("\033[91m¡Algo salió mal, inténtalo de nuevo!")
    raise e

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


¡Perfilado completado!



<br>
<details>
<summary><span style="color:green;"><strong>Solution (Click here to expand)</strong></span></summary>

```python
# Start the second diagnostic run with the new, streamlined model.
trainer_efficient.fit(model_efficient, dm_loader) ### Add your code here

print("\nProfiling Complete!\n")
```

### Paso 4: Análisis de los resultados del perfilador eficiente

* Ejecuta la siguiente celda para mostrar el nuevo informe del perfilador.

In [20]:
# Mostrar las 10 operaciones más costosas en tiempo del informe del profiler_efficient
helper_utils.display_profiler_logs(profiler_efficient, head=10)

Row,Operation Sequence,Action,Total Time (ms),Calls
1,0,ProfilerStep*,170.306085,10
2,112,[pl][profile][_TrainingEpochLoop].train_dataloader_next,86.581060,10
3,113,enumerate(DataLoader)#_MultiProcessingDataLoaderIter.__next__,86.113891,10
4,125,[pl][profile]run_training_batch,62.651159,10
5,126,[pl][profile][LightningModule]CIFAR10LightningModule.optimizer_step,61.955239,10
6,127,Optimizer.step#AdamW.step,61.661106,10
7,129,[pl][profile][Strategy]SingleDeviceStrategy.training_step,34.615147,10
8,16,[pl][profile][Strategy]SingleDeviceStrategy.backward,19.944006,10
9,130,[pl][module]torch.nn.modules.container.Sequential: model,12.685237,10
10,198,[pl][module]torchmetrics.classification.accuracy.MulticlassAccuracy: train_accuracy,11.035525,10


#### Comparando los resultados

Ahora pasamos a la comparación directa. La siguiente celda generará una tabla resumen que muestra el rendimiento de las operaciones computacionales clave antes y después de haber simplificado la arquitectura del modelo.

* Ejecuta la siguiente celda para visualizar la comparación. Observa la diferencia en el tiempo total de `ProfilerStep*` y cómo ha cambiado el tiempo de ejecución para cada núcleo (kernel) computacional.

In [21]:
# Generate the comparison report.
helper_utils.display_comparison_report(profiler, profiler_efficient)

Operation,Total Time Before (ms),Total Time After (ms)
ProfilerStep*,442.274179,170.306085
aten::convolution_backward,4.037066,3.280677
autograd::engine::evaluate_function: AddmmBackward0,3.754525,4.077061
aten::conv2d,3.324034,2.666140
AddmmBackward0,2.653220,2.899359


<br>

#### Analizando la mejora: El impacto de la simplicidad

La tabla comparativa muestra claramente una ganancia de rendimiento significativa al simplificar la arquitectura del modelo. El cambio más importante es la reducción drástica en el tiempo total de `ProfilerStep*`, lo que significa que el modelo más simple completa un paso de entrenamiento mucho más rápido. Esta aceleración proviene directamente de la reducción del tiempo invertido en núcleos (kernels) computacionales centrales como `aten::conv2d` y sus pasos hacia atrás (backward passes).



Esto demuestra una lección clave en la optimización de modelos: **tu arquitectura debe coincidir con la complejidad de tu conjunto de datos**. Para un dataset sencillo como CIFAR-10, un modelo más pequeño es más eficiente, lo que te permite entrenar más rápido.

#### Una nota sobre los resultados

Puede que te resulte interesante que, si bien el tiempo total de `ProfilerStep*` disminuyó significativamente, el tiempo de kernels individuales como `aten::conv2d` o `aten::convolution_backward` cambió solo ligeramente, e incluso en algunos casos pudo haber aumentado. Esto no es un error; resalta cómo funcionan los cuellos de botella de rendimiento.

* **Antes (Modelo Complejo)**: El modelo tenía tensores muy grandes (por ejemplo, `1024` canales). El manejo de estas formas tan grandes añade mucha sobrecarga en la gestión de memoria y en la lógica del framework *alrededor* de los cálculos centrales. La GPU incluso podría estar subutilizada si la sobrecarga de preparar estos grandes tensores causa pequeños retrasos entre operaciones.

* **Después (Modelo Simple)**: La matemática central para una sola convolución no cambia fundamentalmente, pero los tensores sobre los que opera son mucho más pequeños (por ejemplo, `128` canales). La aceleración masiva proviene de la reducción de la sobrecarga. Con formas de datos más simples de gestionar, el framework puede alimentar las operaciones a la GPU de manera más eficiente, lo que conduce a un tiempo total de paso mucho más rápido, incluso si el tiempo de un kernel específico no cambia drásticamente.

Esencialmente, has eliminado el **cuello de botella de la complejidad del modelo**, permitiendo que todo el pipeline funcione de manera mucho más fluida.

#### Más allá de la complejidad del modelo

Es importante recordar que este es solo un tipo de problema de rendimiento que el perfilador puede ayudar a diagnosticar. En otros escenarios, podrías usarlo para descubrir **cuellos de botella en la carga de datos** (donde la GPU está inactiva esperando datos de la CPU), uso ineficiente de la memoria u otras partes del pipeline que ralentizan tu entrenamiento. La clave es utilizar el perfilador como una herramienta versátil para formular y probar hipótesis sobre cualquier aspecto del rendimiento de tu código.

## Entrenando el modelo eficiente

¡Excelente trabajo! Tu análisis y los cambios realizados han dado frutos. El perfilador ha confirmado que el modelo simplificado es significativamente más rápido por cada paso. Sin embargo, la velocidad es solo la mitad de la historia.

La pregunta final y esencial es: **¿viene esta eficiencia a costa del rendimiento?** Un modelo más rápido solo es útil si aún puede alcanzar una buena precisión. Para responder a esto, ahora ejecutarás un bucle de entrenamiento completo en el `model_efficient` e inspeccionarás sus métricas finales.


In [22]:
efficient_results = helper_utils.run_full_training(model_efficient, dm_loader)

print("\nTraining Complete!\n")
print("Final Training Metrics:")

print(f"\tTraining Accuracy:    {efficient_results['train_accuracy']}%")
print(f"\tValidation Accuracy:  {efficient_results['val_accuracy']}%")

--- Running Training For 5 Epochs ---


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Training Complete!

Final Training Metrics:
	Training Accuracy:    80.92%
	Validation Accuracy:  74.69%


* Finalmente, muestra una tabla resumen para comparar las métricas finales de ambos modelos frente a frente, facilitando la visualización de las compensaciones (trade-offs).

In [23]:
helper_utils.display_metrics_comparison(baseline_results, efficient_results)

Metric,Baseline Model,Efficient Model
Training Accuracy (%),86.65,80.92
Validation Accuracy (%),74.17,74.69


<br>

Ahora, analiza tus resultados en la tabla de arriba.

Compara las métricas de validación de tu modelo eficiente con las de la línea base. Es probable que encuentres que el rendimiento es muy similar. Es común que un modelo eficiente y de tamaño adecuado funcione de manera comparable, y a veces incluso ligeramente mejor, que uno excesivamente complejo en el mismo conjunto de datos.

La conclusión clave es evaluar el equilibrio (trade-off). En este caso, es probable que hayas logrado una mejora significativa en la velocidad de entrenamiento sin comprometer de manera importante la capacidad de generalización de tu modelo. Con más ajuste de hiperparámetros o entrenando durante más épocas, probablemente podrías mejorar estos resultados aún más.

## Conclusión

¡Felicidades por completar el laboratorio! Has navegado con éxito a través de un flujo de trabajo completo para diagnosticar y mejorar el rendimiento de un modelo utilizando **Lightning**.

Has visto de primera mano cómo la estructura de Lightning, a través del `LightningDataModule` y el `LightningModule`, elimina el código repetitivo y organiza tu proyecto. Esta estructura limpia hace que sea mucho más sencillo integrar herramientas potentes como el perfilador.

Más importante aún, has aprendido un enfoque práctico y basado en datos para la optimización. Comenzaste estableciendo una línea base, utilizaste el **perfilador** para formular una hipótesis sobre un cuello de botella de rendimiento, probaste esa hipótesis con una segunda ejecución de perfilado y, finalmente, verificaste que tu modelo más eficiente funcionaba igual de bien.

Para aquellos interesados en explorar los detalles granulares completos, pueden descargar los archivos de traza `.JSON` del directorio `./profiler_output/` y cargarlos en un visor de trazas como la [interfaz de usuario de Perfetto](https://ui.perfetto.dev). Esto proporciona mucha más información que la tabla resumen.

Dominar estas técnicas de perfilado es una habilidad esencial para construir modelos eficientes y escalar hacia modelos y conjuntos de datos más grandes.